In [2]:
from langgraph.graph import StateGraph,END,START
from langgraph.checkpoint.memory import MemorySaver
from typing import TypedDict,Annotated
from langgraph.graph.message import add_messages
from langchain_core.messages import AnyMessage
from langchain_openai import ChatOpenAI

class ChatState(TypedDict):
    response:Annotated[list[AnyMessage],add_messages]

llm=ChatOpenAI(model='gpt-4o-mini')

def node(state:ChatState)->dict:
    message=state["response"]
    response=llm.invoke(message)
    return {"response":[response]}

graph=StateGraph(ChatState)

graph.add_node("node",node)
graph.add_edge(START,"node")
graph.add_edge("node",END)

memory=MemorySaver()
chatbot=graph.compile(checkpointer=memory)

print(f"Chahtbot with memory created")

Chahtbot with memory created


In [6]:
from langchain_core.messages import HumanMessage,AIMessage
config={"configurable": {"thread_id":"user-123"}}

result=chatbot.invoke(
   { "response":[HumanMessage(content="My name is Alice and I love Python programming.")]},config=config
)

print(f"BOT: {result['response'][-1].content}")

result2=chatbot.invoke({"response":HumanMessage(content="What do i love")},config=config)

print(f"BOT: {result2['response'][-1].content}")

result3=chatbot.invoke({"response":HumanMessage(content="Make a Joke about it")},config=config)

print(f"BOT: {result3['response'][-1].content}")


BOT: Hi Alice! It's great to hear that you love Python programming. Python is such a versatile and powerful language. What projects or areas of Python are you currently working on or interested in?
BOT: You mentioned that you love Python programming! Is there anything specific within Python or programming in general that you particularly enjoy, like web development, data analysis, machine learning, or something else?
BOT: Sure! Here's a Python-themed joke for you:

Why do Python programmers prefer using `snake_case`?

Because they can't stand the "camel" in their code! 🐍😂

Hope that brought a smile to your face!


In [7]:
from langchain_core.messages import HumanMessage

config2={"configurable":{"thread_id":"user-1234"}}

result=chatbot.invoke({"response":[HumanMessage(content="My name is Bob and I love JavaScript.")]},config=config2)

print(f"BOT: {result["response"][-1].content}")

result2=chatbot.invoke({"response":[HumanMessage(content="What do i Love")]},config=config2)
print(f"BOT: {result2["response"][-1].content}")

result3=chatbot.invoke({"response":[HumanMessage(content="What do i Love")]},config=config)
print(f"BOT: {result3["response"][-1].content}")

BOT: Hi Bob! It's great to see your enthusiasm for JavaScript. It's such a dynamic and versatile language. If you have any questions, projects in mind, or topics you'd like to discuss related to JavaScript, feel free to share!
BOT: You mentioned that you love JavaScript! If there are other things you're passionate about or topics you'd like to explore, feel free to share!
BOT: You love Python programming! If there's anything specific you'd like to discuss about it—whether it's projects, learning resources, or anything else—let me know!
